In [0]:
%sql
--7
create table cyntexa_dev.sales.gold_inventory_summary as
select product_id, sum(quantity) as units_sold, sum(total_amount) as total_sales,
count(distinct order_id) as number_of_sales
from cyntexa_dev.sales.sales_cleaned
group by product_id
order by total_sales desc

- Extended the Gold layer with inventory summary containing total units sold, total sales and number of sales for each product this helps the inventory team identify product demand and support inventory planning

- This belongs in the gold layer because it is a reusable, business-ready aggregation that can be consistently used by dashboards and other consumers, rather than being repeatedly calculated ad hoc by the inventory team

**8**

Databricks splits into control plane (UI, job scheduling, orchestration managed by Databricks) and data plane (actual compute and data, sitting inside the customers own cloud account) for this pipeline all real work reading files, dedupe, type casting, writes to bronze/silver tables runs in the customers data plane, so sensitive data never leaves their cloud

The control plane only handles metadata, scheduling, and logs, no raw data. For security review: focus most effort on locking down the data plane's network and storage access (firewalls, private networking) since thats where actual data risk lives

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("cyntexa_dev.sales.sales_cleaned")
df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()